# 🏥 Disease Prediction Portfolio Project

Welcome to the third portfolio project for **Logistic Regression**. Here, we apply machine learning to the medical domain to predict the presence of cardiovascular disease.

## 1. Business Problem
Early detection of heart disease can save lives. A hospital wants to create a pre-screening tool. Based on a patient's vitals and test results, the tool will output the probability of heart disease, flagging high-risk patients for immediate cardiologist review.

## 2. Dataset Description
We will generate a synthetic dataset mimicking typical clinical data.

**Features:**
- `Age`: Patient age.
- `Cholesterol`: Serum cholesterol in mg/dl.
- `MaxHeartRate`: Maximum heart rate achieved during exercise.
- `BloodPressure`: Resting blood pressure.
- **Target:** `DiseasePresence` (1 for yes, 0 for no).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

np.random.seed(42)
n_samples = 3000

age = np.random.normal(55, 10, n_samples)
cholesterol = np.random.normal(240, 40, n_samples)
max_hr = np.random.normal(150, 20, n_samples)
bp = np.random.normal(130, 15, n_samples)

# Logic: Older age, high cholesterol, low max heart rate, high BP increase disease probability
z = -15.0 + 0.1 * age + 0.02 * cholesterol - 0.03 * max_hr + 0.05 * bp
prob_disease = 1 / (1 + np.exp(-z))
disease = (np.random.rand(n_samples) < prob_disease).astype(int)

df = pd.DataFrame({
    'Age': age,
    'Cholesterol': cholesterol,
    'MaxHeartRate': max_hr,
    'BloodPressure': bp,
    'DiseasePresence': disease
})

df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
sns.pairplot(df, hue='DiseasePresence', plot_kws={'alpha': 0.3})
plt.show()

## 4. Feature Engineering
Scaling is vital in medical datasets since ranges differ wildly (e.g., Age 20-80 vs Cholesterol 150-400).

In [ ]:
X = df.drop('DiseasePresence', axis=1)
y = df['DiseasePresence']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 5. Model Training & 6. Hyperparameter Tuning

In [ ]:
model = LogisticRegression(C=1.0, max_iter=1000)
model.fit(X_train_scaled, y_train)

## 7. Evaluation & 8. Error Analysis
In medicine, a false negative (telling a sick patient they are fine) is catastrophic. We will lower the threshold to increase Recall.

In [ ]:
y_prob = model.predict_proba(X_test_scaled)[:, 1]
y_pred_default = (y_prob >= 0.5).astype(int)
y_pred_safe = (y_prob >= 0.3).astype(int) # Lower threshold for safety

print("--- DEFAULT THRESHOLD (0.5) ---")
print(classification_report(y_test, y_pred_default))

print("--- SAFE THRESHOLD (0.3) ---")
print(classification_report(y_test, y_pred_safe))

## 9. Visualizations & 10. Business Insights
The hospital wants to know what metrics to watch.

In [ ]:
coefs = pd.DataFrame({'Feature': X.columns, 'Weight': model.coef_[0]})
sns.barplot(x='Weight', y='Feature', data=coefs.sort_values('Weight'), orient='h', palette='coolwarm')
plt.title("Predictors of Heart Disease")
plt.show()

print("Insights:")
print("1. Age and Blood Pressure are the strongest positive predictors of disease.")
print("2. A high Max Heart Rate achieved during stress testing is strongly associated with a healthy heart (negative predictor).")

## 11. Future Improvements
- Gather more features like smoking status, BMI, and fasting blood sugar to improve AUC.